# Phase 2: Discrete Modeling & Prior Estimation
**Giorgi Karazanashvili**

Builds on `Data.py` (Phase 1). Treats each biopsy as a Bernoulli trial, uses the Binomial distribution for sample-level questions, and establishes the empirical prior for the Bayesian analysis in later phases.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer

os.makedirs("outputs", exist_ok=True)

BLUE = "#4C8DF5"
RED  = "#E05252"
BG   = "#F4F6F9"

# Load data (same pipeline as Data.py)
raw = load_breast_cancer()
X   = pd.DataFrame(raw.data, columns=raw.feature_names)
y   = pd.Series(raw.target).map({0: 1, 1: 0}).rename("diagnosis")
df  = pd.concat([X, y], axis=1)

n_total = len(df)
n_M     = int(y.sum())
n_B     = n_total - n_M
print(f"Loaded {n_total} biopsies: {n_B} benign, {n_M} malignant")

## 1. Bernoulli Model
Each biopsy is a Bernoulli trial: $X \sim \text{Bernoulli}(\hat{p})$ where "success" = malignant.

In [ ]:
p_hat = n_M / n_total
q_hat = 1 - p_hat
bernoulli_var = p_hat * q_hat

print(f"p̂ = {n_M}/{n_total} = {p_hat:.6f}")
print(f"E[X] = {p_hat:.4f},  Var[X] = {bernoulli_var:.4f},  SD[X] = {np.sqrt(bernoulli_var):.4f}")

## 2. Binomial Model
If $n$ biopsies are drawn independently, the count of malignant cases $Y \sim \text{Bin}(n, \hat{p})$.

In [ ]:
n_sample = 20
k_thresh = 8
binom_rv = stats.binom(n=n_sample, p=p_hat)

p_at_least_8 = 1 - binom_rv.cdf(k_thresh - 1)
p_exactly_8  = binom_rv.pmf(k_thresh)
E_Y  = n_sample * p_hat
SD_Y = np.sqrt(n_sample * p_hat * q_hat)
k_majority = n_sample // 2 + 1
p_majority = 1 - binom_rv.cdf(k_majority - 1)

print(f"Y ~ Bin({n_sample}, {p_hat:.4f})")
print(f"E[Y] = {E_Y:.2f},  SD[Y] = {SD_Y:.2f}")
print(f"P(Y ≥ {k_thresh})  = {p_at_least_8:.6f}  ({p_at_least_8*100:.2f}%)")
print(f"P(Y = {k_thresh})  = {p_exactly_8:.6f}")
print(f"P(Y ≥ {k_majority}) = {p_majority:.6f}  ({p_majority*100:.2f}%) — majority malignant")

print(f"\nP(Y ≥ 1) by sample size:")
for n_q in [1, 3, 5, 10, 20, 50]:
    p1 = 1 - stats.binom.pmf(0, n=n_q, p=p_hat)
    print(f"  n={n_q:<3}  P={p1:.8f}")

## 3. Prior Estimation
Empirical priors $\pi_M$ and $\pi_B$ for the Bayesian classifier (Phase 4), with confidence intervals.

In [ ]:
se_p   = np.sqrt(p_hat * q_hat / n_total)
z_star = stats.norm.ppf(0.975)
ci_lo  = p_hat - z_star * se_p
ci_hi  = p_hat + z_star * se_p

z2 = z_star ** 2
denom = 1 + z2 / n_total
center = (p_hat + z2 / (2 * n_total)) / denom
margin = (z_star / denom) * np.sqrt(p_hat * q_hat / n_total + z2 / (4 * n_total**2))
wilson_lo = center - margin
wilson_hi = center + margin

print(f"π_M = {p_hat:.6f},  π_B = {q_hat:.6f}")
print(f"95% Wald CI:   ({ci_lo:.4f}, {ci_hi:.4f})")
print(f"95% Wilson CI: ({wilson_lo:.4f}, {wilson_hi:.4f})")

## Figure 4 — Binomial PMF

In [ ]:
k_vals   = np.arange(0, n_sample + 1)
pmf_vals = binom_rv.pmf(k_vals)

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor(BG)

bar_colors = [RED if k >= k_thresh else BLUE for k in k_vals]
ax.bar(k_vals, pmf_vals, color=bar_colors, edgecolor="white", width=0.75, alpha=0.85, zorder=3)

for k, p_k in zip(k_vals, pmf_vals):
    if p_k > 0.005:
        ax.text(k, p_k + 0.003, f"{p_k:.3f}", ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.axvline(k_thresh - 0.5, color=RED, lw=1.5, linestyle="--", alpha=0.7)
ax.text(k_thresh + 0.3, max(pmf_vals) * 0.85, f"P(Y ≥ {k_thresh}) = {p_at_least_8:.4f}",
        fontsize=10, color=RED, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=RED, alpha=0.85))
ax.axvline(E_Y, color="#222222", lw=1.5, linestyle=":", alpha=0.6)
ax.text(E_Y + 0.2, max(pmf_vals) * 0.70, f"E[Y] = {E_Y:.1f}", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.7))

ax.set_xlabel("k  (number of malignant biopsies)", fontsize=11)
ax.set_ylabel("P(Y = k)", fontsize=11)
ax.set_title(f"Figure 4 — Binomial PMF:  Y ~ Bin(n={n_sample}, p̂={p_hat:.4f})\n"
             f"Blue = P(Y < {k_thresh}),  Red = P(Y ≥ {k_thresh})", fontsize=12, fontweight="bold")
ax.set_xticks(k_vals)
ax.set_facecolor("#EAECF0")
ax.grid(True, axis="y", color="white", linewidth=0.7, zorder=0)
ax.set_xlim(-0.7, n_sample + 0.7)
plt.tight_layout()
plt.savefig("outputs/fig4_binomial_pmf.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 5 — Binomial CDF & Survival

In [ ]:
cdf_vals = binom_rv.cdf(k_vals)
sf_at_k  = [1 - binom_rv.cdf(k - 1) if k > 0 else 1.0 for k in k_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle(f"Figure 5 — Binomial CDF & Survival:  Y ~ Bin({n_sample}, {p_hat:.4f})",
             fontsize=12, fontweight="bold", y=1.0)

ax1.step(k_vals, cdf_vals, where="mid", color=BLUE, lw=2.2, zorder=3)
ax1.fill_between(k_vals, cdf_vals, step="mid", alpha=0.15, color=BLUE, zorder=2)
ax1.axhline(0.5, color="gray", lw=1, linestyle="--", alpha=0.6)
median_k = int(binom_rv.median())
ax1.plot(median_k, binom_rv.cdf(median_k), "o", color=RED, ms=8, zorder=4)
ax1.text(median_k + 0.5, binom_rv.cdf(median_k) - 0.06, f"Median = {median_k}",
         fontsize=9, color=RED, fontweight="bold")
ax1.set(xlabel="k", ylabel="P(Y ≤ k)", title="CDF — P(Y ≤ k)")
ax1.set_xticks(k_vals[::2])
ax1.set_facecolor("#EAECF0")
ax1.grid(True, color="white", linewidth=0.7, zorder=0)

bar_colors_sf = [RED if s >= 0.05 else "#AAAAAA" for s in sf_at_k]
ax2.bar(k_vals, sf_at_k, color=bar_colors_sf, edgecolor="white", width=0.75, alpha=0.8, zorder=3)
ax2.axhline(0.05, color="#F5A623", lw=1.5, linestyle="--", alpha=0.8, label="α = 0.05")
ax2.set(xlabel="k", ylabel="P(Y ≥ k)", title="Survival — P(Y ≥ k)")
ax2.set_xticks(k_vals[::2])
ax2.set_facecolor("#EAECF0")
ax2.grid(True, axis="y", color="white", linewidth=0.7, zorder=0)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("outputs/fig5_binomial_cdf_survival.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 6 — Prior Sensitivity
Shows how varying the prior $\pi_M$ shifts the posterior $P(\text{Malignant} \mid x)$ for `concave_points_mean` (highest Cohen's d). This bridges the discrete prior estimation to the Bayesian analysis in Phase 4.

In [ ]:
rename_map = {
    "mean radius": "radius_mean", "mean texture": "texture_mean",
    "mean perimeter": "perimeter_mean", "mean area": "area_mean",
    "mean smoothness": "smoothness_mean", "mean compactness": "compactness_mean",
    "mean concavity": "concavity_mean", "mean concave points": "concave_points_mean",
    "mean symmetry": "symmetry_mean", "mean fractal dimension": "fractal_dimension_mean",
}
df.rename(columns=rename_map, inplace=True)

feat = "concave_points_mean"
benign    = df[df["diagnosis"] == 0][feat]
malignant = df[df["diagnosis"] == 1][feat]
bmu, bsd  = benign.mean(), benign.std()
mmu, msd  = malignant.mean(), malignant.std()
x_range   = np.linspace(min(benign.min(), malignant.min()) - 0.01,
                        max(benign.max(), malignant.max()) + 0.01, 500)

priors = [0.10, 0.25, p_hat, 0.50, 0.70]

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor(BG)

cmap = plt.cm.RdYlBu_r
for i, pi_M in enumerate(priors):
    lM   = stats.norm.pdf(x_range, mmu, msd) * pi_M
    lB   = stats.norm.pdf(x_range, bmu, bsd) * (1 - pi_M)
    post = lM / (lM + lB + 1e-12)
    lbl  = f"π_M = {pi_M:.2f}" + ("  ← empirical prior" if abs(pi_M - p_hat) < 0.001 else "")
    ax.plot(x_range, post, lw=2.2, color=cmap(i / (len(priors) - 1)), label=lbl)

ax.axhline(0.5, color="gray", lw=1, linestyle="--", alpha=0.6, label="Decision boundary")
ax.set(xlabel="Concave Points (mean)", ylabel="P(Malignant | x)", ylim=(0, 1.02))
ax.set_title("Figure 6 — Prior Sensitivity: Effect of π_M on Posterior P(Malignant | x)\n"
             "Feature: concave_points_mean  (highest Cohen's d = 2.33)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9, loc="center right")
ax.set_facecolor("#EAECF0")
ax.grid(True, color="white", linewidth=0.7)
plt.tight_layout()
plt.savefig("outputs/fig6_prior_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()